# Your First Object-Oriented Agent

This short tutorial walks through the core ideas that make NOOA special. We'll start with a small toy example: a `BaristaAgent` that recommends drinks to sleepy customers. Along the way we'll see that a NOOA agent is **just a Python object** — you add tools by adding methods, spin up new agents by instantiating the class, and strongly type its outputs like any regular Python function.

We're not going to sell you on a whole new paradigm — you won't walk away having learned some exotic new way to build software. What you *will* walk away with is plain Python, plus a small sprinkle of magic.

## Prerequisites

Install NOOA from GitHub with [uv](https://docs.astral.sh/uv/):

```bash
uv add "nooa @ git+https://github.com/NVIDIA-NeMo/labs-OO-Agents.git@main"
```

The setup cell below lists several providers. Uncomment the one you want to use. For hosted providers, replace `"your-api-key"` with a real key. NOOA is also compatible with **local inference** (Ollama, vLLM, any OpenAI-compatible endpoint) — those need no API key, just an `api_base`.

## Setup

NOOA works with any LiteLLM-supported model — hosted or local. Pick one below. Replace "your-api-key" with a real key for hosted providers; local providers (Ollama, vLLM) don't need a key — just pass api_base.

In [ ]:
from nooa.unifiedllm.registry import get_llm_client

# model = get_llm_client("claude-haiku-4-5", api_key="your-api-key")                                        # Anthropic
# model = get_llm_client("gpt-5-mini", api_key="your-api-key")                                              # OpenAI
# model = get_llm_client("ollama_chat/qwen3:1.7b", api_base="http://localhost:11434")                       # Ollama (local, no key)
# model = get_llm_client("hosted_vllm/Qwen/Qwen3-1.7B", api_base="http://localhost:8000/v1")                # vLLM (local, no key)
model = get_llm_client("openai/openai/openai/gpt-5.5", api_key="sk-NghOY6FpXTh5sK4Y5fquog", api_base="https://inference-api.nvidia.com/v1")  # NVIDIA hosted (OpenAI-compatible)

## A Barista Agent

Let's define our first agent. In NOOA, you build an agent by subclassing `Agent` — the only required argument is the LLM that will power it. Here is a complete, working agent. It's small enough to read every line.

In [17]:
from nooa import Agent

class BaristaAgent(Agent, llm=model):
    """You are a friendly barista at a small neighborhood cafe."""  # this becomes the agent's system prompt

    async def recommend_drink(self, customer_request: str) -> str:
        """Recommend a single drink to the customer based on what they just told you.
        Be warm and concise — one string sentence is plenty."""
        ...

Instantiate and call it. Notice that we `await` the method — generation methods are async.

In [18]:
barista = BaristaAgent()
result = await barista.recommend_drink("Hi, I feel so sleepy.")
print(result)

Sounds like a cozy latte with an extra shot of espresso would perk you up nicely.


That's it — a friendly recommendation, ready to serve. So what just happened?

Behind the scenes, NOOA does the work that keeps the interface this Pythonic. A few things worth noticing:

- the **class docstring** became the agent's system prompt
- the **method docstring** became the task description
- the **ellipsis (`...`)** is how you tell NOOA "this method is *agentic* — hand it off to the LLM instead of running it as regular Python"

Concretely, NOOA quietly assembles a prompt that looks roughly like:

```
<system prompt>
You are a friendly barista at a small neighborhood cafe.

<available methods>
recommend_drink(customer_request: str) -> str
Recommend a single drink to the customer based on what they just told you. Be warm and concise — one sentence is plenty.
</available methods>
```

Plus a few other pieces we'll unpack later. Let's call the agent one more time, just for fun:

In [19]:
result = await barista.recommend_drink("I would love something that works with a cornetto.")
print(result)

A smooth cappuccino would pair beautifully with your cornetto.


> 📝 **Takeaway:** the agent is an object.

## Adding Tools

The barista just offered a strong caffeine kick — but it's 9pm and a double espresso is definitely not the move. How do we teach the agent to respect a "no caffeine after 4pm" policy?

In most agent frameworks, you'd register a tool, describe it in a JSON schema, and wire it into the runtime. In NOOA, you just **add a method to the class**. Anything the agent can see on itself, it can call as a tool.

Let's add our first tool to `BaristaAgent`.

In [22]:
from datetime import datetime
from nooa import Agent

class BaristaAgent(Agent, llm=model):
    """You are a friendly barista at a small neighborhood cafe."""

    def is_only_decaf_hour(self) -> bool:
        """Return True if we should only be serving decaf right now. After 2pm we go decaf-only so our customers can still sleep tonight."""
        return datetime.now().hour >= 14

    async def recommend_drink(self, customer_request: str) -> str:
        """Recommend a single drink to the customer based on what they just told you.
        Be warm and concise — one string sentence is plenty."""
        ...

barista = BaristaAgent()
await barista.recommend_drink("Hi, I feel so sleepy.")

"Sounds like a cozy pick-me-up moment—I'd recommend a decaf latte, warm and comforting without keeping you up later."

> 📝 **Takeaway:** in NOOA, ordinary Python methods and agentic methods live side by side on the same class. The agent freely calls the deterministic ones as tools — no registration, no schema, no glue code.

## Strong Typing

What if our cafe only serves a fixed menu, and we want the agent to *only* recommend drinks we actually offer?

Most agent frameworks deal in a single data type: text. Text gets passed to tools, text gets exchanged between agents, text comes back as output — and then you painstakingly parse it into JSON, hoping the model got the shape right (which, even with the best models, it doesn't always). NOOA takes a different approach: because the agent lives inside a Python program, we can strongly type everything.

Let's put a real type on the return value of `recommend_drink`:

In [ ]:
from enum import Enum
from nooa import Agent


class Drink(Enum):
    ESPRESSO = "espresso"
    CAPPUCCINO = "cappuccino"
    FLAT_WHITE = "flat white"

class BaristaAgent(Agent, llm=model):
    """You are a friendly barista at a small neighborhood cafe."""

    def is_only_decaf_hour(self) -> bool:
        """Return True if we should only be serving decaf right now. After 4pm we go decaf-only so our customers can still sleep tonight."""
        return datetime.now().hour >= 16

    async def recommend_drink(self, customer_request: str) -> tuple[str, Drink]:
        """Pick the single best drink for the customer from the menu, based on what they told you."""
        ...

barista = BaristaAgent()
reason, drink = await barista.recommend_drink("Hi, I feel so sleepy.")
print("Barista: ", reason)
print("Recommended drink: ", drink)
print(type(drink))

Barista:  You sound like you could use a quick pick-me-up — I’d recommend an espresso.
Recommended drink:  Drink.ESPRESSO
<enum 'Drink'>


This isn't just prompt engineering — NOOA enforces the return type at runtime. If the LLM returns something that isn't a valid `Drink`, the framework retries until it produces one. You get real Python objects back, not strings you have to reparse.

> 📝 **Takeaway:** strong typing all the way through.

## Give Your Agent Some State

Our cafe has a finite stash of coffee beans, and every drink burns through a few. Since our agent is *just a Python object*, giving it state is as easy as adding a field in `__init__`:

In [15]:
from nooa import Agent
from typing import Union

class Drink(Enum):
    ESPRESSO = "espresso"
    CAPPUCCINO = "cappuccino"
    FLAT_WHITE = "flat white"
    TEA = "tea"

class BaristaAgent(Agent, llm=model):
    """You are a friendly barista at a small neighborhood cafe."""

    def __init__(self, coffee_beans: int) -> None:
        super().__init__()
        self.coffee_beans = coffee_beans

    def is_only_decaf_hour(self) -> bool:
        """Return True if we should only be serving decaf right now. After 4pm we go decaf-only so our customers can still sleep tonight."""
        return datetime.now().hour >= 16

    def serve(self, drink: Drink) -> Drink:
        """Serve a drink. Deducts 100 beans unless it's tea."""
        if drink != Drink.TEA:
            self.coffee_beans -= 100
        return drink

    async def recommend_drink(self, customer_request: str) -> tuple[str, Union[Drink, None]]:
        """Recommend a single drink to the customer based on what they just told you.
        Be warm and concise — one sentence is plenty.
        We currently have {self.coffee_beans} beans left. If we ran out, recommend a tea.
        Call self.serve(drink) with your chosen drink before returning."""
        ...

In [16]:
barista = BaristaAgent(coffee_beans=200)
for _ in range(3):
    barista_answer, drink = await barista.recommend_drink("Something to keep me going, please.")
    print(f"Answer: {barista_answer}\nServed: {drink}\nBeans left: {barista.coffee_beans}")

Answer: Absolutely — an espresso should give you a nice little boost.
Served: Drink.ESPRESSO
Beans left: 100
Answer: Absolutely — an espresso should give you a bright little boost.
Served: Drink.ESPRESSO
Beans left: 0
Answer: We’re out of beans, but I’d be happy to make you a comforting tea to keep you going.
Served: Drink.TEA
Beans left: 0


> 📝 **Takeaway:** the agent "sees" everything on itself — including itself.

## Peeking Inside the Context

When you write an agentic method, NOOA's harness builds the real prompt for you and makes sure the LLM sees exactly what it needs — and nothing more. Curious what the agent actually sees? Two helpers to keep in your back pocket:

- `doc(self)` — the auto-generated API view of your agent (methods, docstrings, types). This is the same view the LLM gets.
- `print_prompt(agent.method, ...)` — the fully rendered prompt for a specific method call, template variables and all.

## What Is Happening Under the Hood?

NOOA doesn't hide the prompt from you. `print_prompt` renders exactly what would be sent to the LLM for a given method call — the system prompt, the agent introspection block, and the task. Take a look:

In [19]:
import nooa
await nooa.print_prompt(barista.recommend_drink, mood="stressed and running late")

=== SYSTEM PROMPT  [BaristaAgent] ===

<system_prompt expr="self._resolve_system_prompt()">
You are a friendly barista at a small neighborhood cafe.
</system_prompt>

<strategy_prompt>
## Strategy

Jupyter-like Python session. Parameters pre-loaded as locals; state persists across cells. Use `await` directly, `print`/`pprint` to debug, `doc(obj)` to inspect types. You MUST call a tool each turn — **plain-text responses do NOT end the session**. To finish, call `return_result(value)`. Repeated text-only responses will abort the run with an error.

**Your two tools:**
- `execute_python(code)` — run a code cell
- `return_result(value)` — submit your final answer (also callable from inside `execute_python`)

## When to use which tool

Use `return_result(...)` directly for simple answers determinable from the inputs alone (yes/no, one field, a single lookup).

Use `execute_python(...)` for lists/batches, arithmetic, multi-step computation, transforms, or iteration. Always iterate in code — 

Look through the output. There's no hidden state — this is exactly what the LLM sees. A few blocks worth naming:

- **`<system_prompt>`** — opens with your class docstring. This is the persona the model wears for every method on the class.
- **`<strategy_prompt>`** — a compact rulebook for how to act each turn: what tools exist (`execute_python`, `return_result`), when to use which, and how to finish a run. This is what teaches the LLM to "inhabit" the framework, and it's the same for every agent.
- **`<execution_context>`** — the imports, types, and helpers that will be in scope when the LLM writes code. Anything you import at module level shows up here.
- **`<self>`** — auto-generated documentation of the agent's public methods and fields, rendered from `doc(type(self))`. This is how the LLM discovers what the agent can do — no separate tool registry needed.
- **Task prompt** — your method docstring, with parameters like `mood` bound to the values you passed in. Rendered live at call time.

That's the whole prompt. No hidden system messages, no template files, no per-tool JSON schemas glued on the side. The reason the "rulebook" stays short is that the framework is plain Python, and the model already knows Python.

> **Tip:** `print_prompt` only shows the *outgoing* prompt. To watch a whole run unfold — the LLM's response, generated code, tool calls, retries, validation — the framework ships a live **trace viewer** (`nooa start-dev`, served on `localhost:5001`). Any agent you create will stream into it automatically once it's running.

### Where the Magic Happens

Two things to sit with:

- **You didn't register `is_only_decaf_hour` anywhere.** No `@tool`, no JSON schema, no `tools=[...]` list. The framework rendered `doc(self)` into the system prompt (that `<self>` block you saw earlier), and the LLM discovered the method the same way you'd discover it while reading someone else's code.
- **The generation method used the helper without being told to.** Nothing in `recommend_drink`'s docstring mentions `is_only_decaf_hour`. The LLM spotted a method that looked relevant, called it, and folded the result into its recommendation. That's the whole "tools are just methods" idea in a single page.

### No Tool Registration

> **Why this is different:** every other agent framework asks you to register tools — usually a name, a JSON schema for arguments, a description string, sometimes a return schema. Here you just define a method. Type hints become the schema. The docstring becomes the description. Python does the work that other frameworks pile on top of it.

The corollary: renaming a method renames the tool. Adding a parameter changes the tool signature. Deleting a method removes the tool. Refactor the class like you'd refactor any other Python class.

## Recap

Five things the barista taught us:

- **Ellipsis `...` marks a generation method.** No decorator, no separate registry. If the body is `...`, the LLM implements it.
- **The class docstring is the system prompt; the method docstring is the task.** Rewriting a prompt means editing a docstring.
- **`{self.attr}` in docstrings is live.** Change the attribute, and the next call sees the new value.
- **Every non-hidden method on `self` is a tool.** No `@tool` decorator, no JSON schema, no registration step.
- **The return type annotation is the output contract.** Pydantic models are validated and retried automatically.

## Exercises

Try these in a fresh cell. Solutions are one small edit each.

1. **Daily special.** Add a `daily_special: str` field to `BaristaAgent.__init__` and reference it in the `recommend_drink` docstring with `{self.daily_special}`. Confirm the LLM starts pushing that drink.
2. **Refund method.** Add a `refund(self, drink: str) -> str` generation method that produces the barista's grudging apology and refund line.
3. **Time of day.** Add a `time_of_day: Literal["morning", "afternoon", "closing"]` field to the return type and rerun. Confirm it appears in the output — no other code change required.
4. **Bean anxiety.** Modify the `recommend_drink` docstring so the barista is documented as noticeably snarkier when `self.coffee_beans < 5`. Set `barista.coffee_beans = 2` and observe the snark level rise.